<a href="https://colab.research.google.com/github/rommeljose/RAG_Local_Google_Colab/blob/main/RAG_AGUA.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

## Mi RAG de ejemplo en Colab

In [ ]:
import subprocess
import time
import os

# Install zstd, which is required by Ollama installer
!sudo apt-get update
!sudo apt-get install -y zstd

# 1. Instalamos Ollama
!curl -fsSL https://ollama.com/install.sh | sh

# 2. Encendemos el servidor internamente desde Python y le damos 5 segundos para arrancar
print("Iniciando el servidor de Ollama en segundo plano...")
# Kill any existing ollama process to avoid conflicts
# The `|| true` prevents the command from failing if no ollama process is found
os.system("killall ollama || true")
time.sleep(1) # Give it a moment to kill

# Start ollama serve in the background using nohup and redirect output
# This keeps the process running even if the Colab cell finishes execution
os.system("nohup ollama serve > ollama.log 2>&1 &")
time.sleep(5) # Give it some time to start

print("¡Servidor listo! Ya puedes continuar con la siguiente celda.")
print("Verifica el archivo 'ollama.log' para el estado del servidor si hay problemas.")

Aquí instalamos el mismo requirements.txt que usamos en nuestra prueba usando pip.

In [ ]:
!pip install -q -U langchain langchain-chroma langchain-ollama langchain-community chromadb pypdf beautifulsoup4

Usamos la terminal de Colab para decirle al servidor fantasma de Ollama que empiece a descargar la IA usando nuestro gran y privilegiado ancho de banda de Google.

In [ ]:
!ollama pull nomic-embed-text

Previamente, a la izquierda de la interfaz gráfica de Colab existe la pestaña de Archivos (ícono de carpeta). Allí podrás usar el drag&drop para soltar tu informacion

In [ ]:
import os
from langchain_community.document_loaders import PyPDFDirectoryLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_ollama import OllamaEmbeddings
from langchain_chroma import Chroma

# Crearemos una carpeta artificial y moveremos ahí PDFs subidos sueltos a colab
os.makedirs("documentos", exist_ok=True)
!mv *.pdf documentos/ 2>/dev/null

loader = PyPDFDirectoryLoader("documentos")
documents = loader.load()

text_splitter = RecursiveCharacterTextSplitter(chunk_size=500, chunk_overlap=50)
chunks = text_splitter.split_documents(documents)

embeddings = OllamaEmbeddings(model="nomic-embed-text")
db = Chroma.from_documents(chunks, embeddings, persist_directory="chroma_db")
print("✅ Ingesta Lista en Colab.")


 Hacer Consultas Extremadamente Rápidas con GPU
Finalmente pegamos nuestro mismo código limpio para buscar y consultar al LLM.

In [ ]:
from langchain_chroma import Chroma
from langchain_ollama import OllamaEmbeddings, ChatOllama
from langchain_core.prompts import ChatPromptTemplate
import time

query_text = '¿Qué es la toma en el Camcamure y cuanta agua suministra?'

embeddings = OllamaEmbeddings(model="nomic-embed-text")
db = Chroma(persist_directory="chroma_db", embedding_function=embeddings)
results = db.similarity_search(query_text, k=50)
context_text = "\n\n---\n\n".join([doc.page_content for doc in results])

SYSTEM_PROMPT = """Eres el Especialista Técnico en Hidráulica de Cumaná.
Tu misión es extraer, resumir y estructurar la información solicitada utilizando estrictamente el Contexto provisto.

Reglas obligatorias:
1. Idioma: ESPAÑOL estricto.
2. Tono: Puramente técnico.
3. Restricción: Responde ÚNICAMENTE con los datos del "Contexto"."""

prompt_template = ChatPromptTemplate.from_messages([
    ("system", SYSTEM_PROMPT),
    ("user", "Contexto recuperado:\n{context}\n\nPregunta: {question}")
])
prompt = prompt_template.invoke({"context": context_text, "question": query_text})

model = ChatOllama(model="llama3.1")

start_time = time.time()
response = model.invoke(prompt)
print(f"Tiempo de hardware Google T4: {time.time() - start_time:.2f} s")
print("-" * 50)
print(response.content)


### Modelos y Parámetros Utilizados en este Sistema RAG

Para el sistema de Recuperación Aumentada por Generación (RAG) implementado, se han utilizado dos modelos clave a través de la plataforma Ollama:

1.  **Modelo de Embedding: `nomic-embed-text`**
    *   **Función:** Este modelo es el encargado de transformar tanto los documentos de referencia (tus PDFs cargados) como las consultas del usuario en representaciones numéricas (vectores o *embeddings*). Estos vectores capturan el significado semántico del texto.
    *   **Importancia:** Permite al sistema encontrar rápidamente los fragmentos de texto más relevantes de tu base de conocimientos que son semánticamente similares a la pregunta del usuario. Sin un buen modelo de embedding, la recuperación de información sería ineficaz.
    *   **Parámetros:** Al ser un modelo pre-entrenado, su arquitectura y pesos internos son fijos. Los parámetros principales son la dimensión del espacio vectorial de los embeddings (determinada por el modelo), y cómo se manejan los "chunks" (fragmentos de texto) que se configuran en el `RecursiveCharacterTextSplitter`.

2.  **Modelo de Inferencia (LLM): `llama3.1`**
    *   **Función:** Una vez que el `nomic-embed-text` ha recuperado el contexto más relevante, el modelo `llama3.1` entra en acción. Su tarea es leer la pregunta del usuario y el contexto recuperado para generar una respuesta coherente, relevante y con el tono especificado en el `SYSTEM_PROMPT`.
    *   **Importancia:** `llama3.1` es un modelo de lenguaje grande (LLM) de Meta, conocido por su capacidad de razonamiento y generación de texto de alta calidad. Su uso asegura que la respuesta final no solo sea precisa (basada en el contexto), sino también bien redactada y fácil de entender.
    *   **Parámetros:** Como LLM, tiene millones o miles de millones de parámetros entrenados. En este caso, se invoca con la configuración por defecto proporcionada por Ollama para `llama3.1`, que ya está optimizada para buen rendimiento y calidad. Otros parámetros comunes para los LLMs incluyen la temperatura (creatividad de la respuesta), `top_p` (muestreo de tokens), etc., aunque aquí se utilizan los valores predeterminados de la implementación de LangChain/Ollama.

**Flujo del RAG:**
La combinación de `nomic-embed-text` para una recuperación precisa y `llama3.1` para una generación inteligente permite que tu sistema RAG responda preguntas complejas utilizando tu propia información, superando las limitaciones de conocimiento de un LLM base y proporcionando respuestas **fundamentadas en tus documentos**.

¡Todo es idéntico a nivel de código Python y LangChain! La única gran diferencia para mudar este entorno entero hacia Colab es la forma de instalar la aplicación Ollama (Celda 1) para evitar depender de instalación gráfica, y correrlo de fondo para no trabar la celda donde escribes.

## Estrategias para Mejorar tu Sistema RAG

Dado que ya tienes un sistema RAG funcionando bien, aquí hay varias áreas en las que podrías enfocarte para optimizar aún más su rendimiento:

1.  **Optimización del Procesamiento de Documentos (Chunking):**
    *   **Chunking Semántico:** En lugar de dividir los documentos por tamaño fijo, explora métodos que dividan el texto basándose en la coherencia semántica para asegurar que cada 'chunk' contenga información completa sobre un tema. Herramientas como `LangChain` o `LlamaIndex` ofrecen diferentes `text_splitters` que podrías probar.
    *   **Chunking Jerárquico/Anidado:** Divide los documentos en chunks de diferentes tamaños y niveles de detalle, permitiendo que el recuperador elija el nivel de granularidad más apropiado para la consulta.
    *   **Contexto Adicional en Metadata:** Añade más metadatos relevantes a cada chunk (título del documento, sección, autor, etc.) que puedan ayudar al LLM a entender mejor el contexto.

2.  **Mejora del Modelo de Embedding:**
    *   **Modelos Específicos del Dominio:** Si `nomic-embed-text` no fue entrenado específicamente para tu dominio (ej. hidráulica, ingeniería), podrías buscar o incluso _fine-tunear_ un modelo de embedding con tus propios datos para obtener representaciones vectoriales más precisas.
    *   **Evaluación de Embeddings:** Mide qué tan bien tu modelo de embedding diferencia entre documentos relevantes e irrelevantes. Métricas como la precisión de recuperación `@k` pueden ser útiles.

3.  **Técnicas de Recuperación Avanzadas:**
    *   **Re-ranking:** Después de obtener los top-K documentos con la búsqueda de similitud, utiliza un modelo de re-ranking (generalmente un modelo Transformer más pequeño o un LLM) para reordenar esos documentos. Esto puede mejorar significativamente la relevancia de los documentos finales enviados al LLM.
    *   **Búsqueda Híbrida:** Combina la búsqueda vectorial (similitud semántica) con la búsqueda por palabras clave (ej. BM25) para capturar tanto la relevancia semántica como la lexical.
    *   **Multi-vector Retrieval:** En lugar de un solo vector por chunk, puedes generar múltiples vectores para diferentes aspectos del mismo chunk (ej. un vector para el título, otro para el resumen, otro para el contenido completo).

4.  **Optimización del LLM (Generación):**
    *   **Prompt Engineering Avanzado:** Experimenta con prompts más complejos, incluyendo instrucciones más detalladas, ejemplos `few-shot`, o técnicas como `Chain-of-Thought` para guiar al LLM a producir respuestas más precisas y estructuradas.
    *   **Modelos LLM Especializados:** Prueba con otros modelos de `Ollama` o incluso con APIs externas si la precisión o el estilo de `llama3.1` no son suficientes.
    *   **Parámetros de Generación:** Ajusta parámetros como `temperature`, `top_p`, `max_new_tokens` para controlar la creatividad y la longitud de las respuestas del LLM.

5.  **Evaluación Continua:**
    *   **Métricas Cuantitativas:** Establece un conjunto de preguntas de prueba y evalúa sistemáticamente la precisión, la completitud, la fidelidad (que no alucine) y la relevancia de las respuestas de tu RAG.
    *   **Evaluación Humana:** Involucra a expertos del dominio para evaluar las respuestas, lo cual es invaluable para captar matices que las métricas automáticas pueden pasar por alto.
    *   **Monitoreo de Fallos:** Registra las consultas donde el RAG falla o produce respuestas de baja calidad para identificar patrones y áreas de mejora.

6.  **Manejo de Consultas Complejas:**
    *   **Query Expansion/Rewriting:** Para consultas complejas o ambiguas, el sistema puede reescribir o expandir la consulta antes de enviarla al recuperador para obtener mejores resultados.
    *   **Multi-hop Reasoning:** Para preguntas que requieren integrar información de múltiples partes del documento o de varios documentos, puedes implementar un sistema que realice múltiples pasos de recuperación y generación.

Empezar por el **re-ranking** o ajustar el **chunking** suelen ser puntos de partida con un alto retorno de inversión en términos de mejora de rendimiento.

### Implementando Re-ranking para Mejorar la Relevancia

El **re-ranking** es una técnica post-recuperación que toma los documentos iniciales recuperados por tu sistema de similitud (en tu caso, `Chroma.similarity_search`) y los reordena basándose en una evaluación más profunda de su relevancia para la consulta. Esto es especialmente útil porque los modelos de embedding pueden recuperar documentos que son semánticamente similares, pero un re-ranker puede refinar esa lista priorizando aquellos que son más directamente pertinentes.

Un método común es usar un *cross-encoder* (codificador cruzado). A diferencia de los *bi-encoders* (como `nomic-embed-text`) que codifican la consulta y los documentos de forma independiente, un cross-encoder procesa la consulta y cada documento como un par, lo que le permite capturar interacciones más finas y complejas entre ellos para determinar una puntuación de relevancia más precisa.

Aquí demostraremos cómo integrar un re-ranker simple utilizando `SentenceTransformers` que tiene modelos cross-encoder pre-entrenados.

In [ ]:
# Instalar la librería SentenceTransformers si aún no está instalada
!pip install -q  sentence-transformers

import os
from langchain_community.document_loaders import PyPDFDirectoryLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_ollama import OllamaEmbeddings, ChatOllama
from langchain_chroma import Chroma
from langchain_core.prompts import ChatPromptTemplate
import time

from sentence_transformers import CrossEncoder

print("Librerías y re-ranker importados correctamente.")

In [ ]:
# Cargar el modelo CrossEncoder para re-ranking
# 'cross-encoder/ms-marco-TinyBERT-L-2' es un modelo ligero y eficaz para tareas de re-ranking
# Puedes experimentar con otros modelos de Hugging Face si necesitas más precisión y tienes más recursos.
reranker_model = CrossEncoder('cross-encoder/ms-marco-TinyBERT-L-2')

# Consulta de ejemplo
query_text_rerank = '¿Qué es la toma en el Camcamure y cuanta agua suministra?'

# Paso 1: Recuperación inicial (como ya lo haces)
embeddings = OllamaEmbeddings(model="nomic-embed-text")
db = Chroma(persist_directory="chroma_db", embedding_function=embeddings)

# Recuperar un número mayor de documentos iniciales (e.g., k=20) antes del re-ranking
initial_results = db.similarity_search(query_text_rerank, k=20)

print(f"Documentos iniciales recuperados: {len(initial_results)}")

# Preparar pares de consulta-documento para el re-ranker
sentence_pairs = [[query_text_rerank, doc.page_content] for doc in initial_results]

# Paso 2: Re-ranking de los documentos
rerrank_scores = reranker_model.predict(sentence_pairs)

# Asociar las puntuaciones con los documentos originales y ordenarlos
reranked_docs = []
for score, doc in zip(rerrank_scores, initial_results):
    doc.metadata['rerank_score'] = float(score) # Añadir la puntuación como metadato
    reranked_docs.append(doc)

# Ordenar los documentos por la puntuación de re-ranker (descendente)
reranked_docs_sorted = sorted(reranked_docs, key=lambda x: x.metadata['rerank_score'], reverse=True)

# Seleccionar los top-k' documentos re-rankeados (e.g., top 5 para el contexto del LLM)
final_context_docs = reranked_docs_sorted[:5]

# Crear el contexto para el LLM con los documentos re-rankeados
context_text_reranked = "\n\n---\n\n".join([doc.page_content for doc in final_context_docs])

# Paso 3: Generación con el LLM (como ya lo haces)
SYSTEM_PROMPT_RERANK = """Eres el Especialista Técnico en Hidráulica de Cumaná.
Tu misión es extraer, resumir y estructurar la información solicitada utilizando estrictamente el Contexto provisto.

Reglas obligatorias:
1. Idioma: ESPAÑOL estricto.
2. Tono: Puramente técnico.
3. Restricción: Responde ÚNICAMENTE con los datos del "Contexto"."""

prompt_template_rerank = ChatPromptTemplate.from_messages([
    ("system", SYSTEM_PROMPT_RERANK),
    ("user", "Contexto recuperado:\n{context}\n\nPregunta: {question}")
])
prompt_rerank = prompt_template_rerank.invoke({"context": context_text_reranked, "question": query_text_rerank})

model = ChatOllama(model="llama3.1")

start_time = time.time()
response_reranked = model.invoke(prompt_rerank)
print(f"\nTiempo de hardware Google T4 (con re-ranking): {time.time() - start_time:.2f} s")
print("-" * 50)
print(response_reranked.content)

print("\n--- Primeros 3 documentos con sus puntuaciones de re-ranking ---")
for doc in final_context_docs[:3]:
    print(f"Score: {doc.metadata['rerank_score']:.4f}, Contenido: {doc.page_content[:200]}...")

Al comparar la respuesta generada con y sin re-ranking, deberías notar una mejora en la especificidad y precisión, ya que el LLM ahora recibe un contexto más concentrado y relevante.

### Evaluación Automatizada de Calidad de Respuestas (LLM-as-a-Judge)

Ahora, implementaremos una evaluación automatizada utilizando un LLM como 'juez'. Evaluaremos dos métricas clave para cada respuesta:

1.  **Fidelidad (Faithfulness):** ¿La respuesta generada está completamente soportada por la información del contexto proporcionado?
2.  **Relevancia (Relevance):** ¿La respuesta generada aborda directamente la pregunta del usuario?

Usaremos el mismo modelo `llama3.1` (o uno similar si prefieres) para realizar esta evaluación.

In [ ]:
from langchain_ollama import ChatOllama
from langchain_core.prompts import ChatPromptTemplate

# Inicializar el modelo para la evaluación (puede ser el mismo que el de generación)
eval_model = ChatOllama(model="llama3.1")

def evaluate_response(query, context, response_content, llm_judge):
    # Prompt para evaluar la Fidelidad
    faithfulness_prompt_template = ChatPromptTemplate.from_messages([
        ("system", "Eres un evaluador imparcial. Tu tarea es determinar si la respuesta generada es completamente fiel al contexto proporcionado. Responde SOLO con 'Sí' o 'No'."),
        ("user", "Contexto:\n{context}\n\nRespuesta Generada:\n{response_content}\n\n¿La respuesta es completamente fiel al contexto?")
    ])
    faithfulness_check = llm_judge.invoke(faithfulness_prompt_template.invoke({"context": context, "response_content": response_content}))

    # Prompt para evaluar la Relevancia
    relevance_prompt_template = ChatPromptTemplate.from_messages([
        ("system", "Eres un evaluador imparcial. Tu tarea es determinar si la respuesta generada es relevante para la pregunta del usuario. Responde SOLO con 'Sí' o 'No'."),
        ("user", "Pregunta del Usuario:\n{query}\n\nRespuesta Generada:\n{response_content}\n\n¿La respuesta es relevante para la pregunta?")
    ])
    relevance_check = llm_judge.invoke(relevance_prompt_template.invoke({"query": query, "response_content": response_content}))

    return faithfulness_check.content.strip(), relevance_check.content.strip()

print("Evaluando respuesta SIN re-ranking...")
faith_orig, relev_orig = evaluate_response(query_text, context_text, response.content, eval_model)
print(f"  - Fidelidad (Sin Re-ranking): {faith_orig}")
print(f"  - Relevancia (Sin Re-ranking): {relev_orig}")

print("\nEvaluando respuesta CON re-ranking...")
faith_rerank, relev_rerank = evaluate_response(query_text_rerank, context_text_reranked, response_reranked.content, eval_model)
print(f"  - Fidelidad (Con Re-ranking): {faith_rerank}")
print(f"  - Relevancia (Con Re-ranking): {relev_rerank}")